In [ ]:
import os
DATA_DIR = os.getenv('DATA_DIR', '/path/to/local/private/data')
SYS_ORDER_FILE = os.getenv('SYS_ORDER_FILE', 'private_sys_order.xlsx')
GEO_FILE = os.getenv('GEO_FILE', 'private_geo.xlsx')
GOOGLE_MAPS_KEY = os.getenv('GOOGLE_MAPS_KEY', '')


In [ ]:
# Install necessary packages
%pip install pandas openpyxl matplotlib seaborn geopy numpy pyproj math folium requests


##### Load in the Data Set
###### Note: Ensure the notebook and Excel files are in the same directory.
###### Open the notebook in VS Code.
###### Run the first cell to install the required packages.
###### Run the subsequent cells to load and display the data.


In [ ]:
# Importing the necessary libraries
import pandas as pd

# Function to load datasets
def load_data():
    # Load the datasets from the provided file paths
    order_data_path = os.path.join(DATA_DIR, SYS_ORDER_FILE)
    geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
    
    # Read the Excel files
    order_data = pd.read_excel(order_data_path, sheet_name=None)
    geo_data = pd.read_excel(geo_data_path)
    
    # Accessing specific sheets in <private-order-file>.xlsx
    order_header = order_data['order_header']
    order_line = order_data['order_line']
    
    return order_header, order_line, geo_data

# Load the data
order_header, order_line, geo_data = load_data()

# Display the first few rows of each dataframe to ensure correct loading
print(order_header.head())



In [ ]:
print(order_line.head())


In [ ]:
print(geo_data.head())

##### Clean Data:

###### Handle missing values and ensure data types are correct.


In [ ]:
# Clean the data: handle missing values and correct data types
def clean_data(order_header, order_line, geo_data):
    # Check for missing values and fill or drop them as necessary
    order_header = order_header.dropna(subset=['status', 'consignment', 'ship_by_date', 'is_vas', 'carrier_id', 'customer_id_slim'])
    order_line = order_line.dropna(subset=['order_id', 'full_pallets'])
    geo_data = geo_data.dropna()
    
    # Ensure correct data types using .loc to avoid SettingWithCopyWarning
    order_header.loc[:, 'ship_by_date'] = pd.to_datetime(order_header['ship_by_date'])
    
    return order_header, order_line, geo_data

# Clean the data
order_header, order_line, geo_data = clean_data(order_header, order_line, geo_data)

# Display cleaned data info
print("Cleaned Order Header Data Info:")
print(order_header.info())



In [ ]:
print("\nCleaned Order Line Data Info:")
print(order_line.info())


In [ ]:

print("\nCleaned Geo Data Info:")
print(geo_data.info())

##### Additional Data Cleaning 
###### Remove Duplicates: Ensure there are no duplicate entries in the datasets.
###### Validate Values: Ensure that values fall within expected ranges or categories.
###### Handle Inconsistent Data: Standardize formats for columns like postcode and customer_id.

In [ ]:
# Step 1: Remove duplicates from the datasets
def remove_duplicates(order_header, order_line, geo_data):
    order_header = order_header.drop_duplicates()
    order_line = order_line.drop_duplicates()
    geo_data = geo_data.drop_duplicates()
    return order_header, order_line, geo_data

# Apply step 1
order_header, order_line, geo_data = remove_duplicates(order_header, order_line, geo_data)

# Display cleaned data info after removing duplicates
print("Order Header Data Info after Removing Duplicates:")
print(order_header.info())

print("\nOrder Line Data Info after Removing Duplicates:")
print(order_line.info())

print("\nGeo Data Info after Removing Duplicates:")
print(geo_data.info())


In [ ]:
# Step 2: Validate values in the datasets
def validate_values(order_header, order_line, geo_data):
    # Example validation: ensuring 'full_pallets' is non-negative
    order_line = order_line[order_line['full_pallets'] >= 0]
    return order_header, order_line, geo_data

# Apply step 2
order_header, order_line, geo_data = validate_values(order_header, order_line, geo_data)

# Display cleaned data info after validating values
print("Order Header Data Info after Validating Values:")
print(order_header.info())

print("\nOrder Line Data Info after Validating Values:")
print(order_line.info())

print("\nGeo Data Info after Validating Values:")
print(geo_data.info())


In [ ]:
# Step 3: Handle inconsistent data formats
def standardize_formats(order_header, order_line, geo_data):
    # Standardize formats for 'postcode' and 'customer_id'
    order_header['postcode'] = order_header['postcode'].str.upper().str.replace(' ', '')
    order_line['customer_id'] = order_line['customer_id'].str.upper().str.strip()
    geo_data['POSTCODE'] = geo_data['POSTCODE'].str.upper().str.replace(' ', '')
    geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.upper().str.strip()
    return order_header, order_line, geo_data

# Apply step 3
order_header, order_line, geo_data = standardize_formats(order_header, order_line, geo_data)

# Display cleaned data info after standardizing formats
print("Order Header Data Info after Standardizing Formats:")
print(order_header.info())

print("\nOrder Line Data Info after Standardizing Formats:")
print(order_line.info())

print("\nGeo Data Info after Standardizing Formats:")
print(geo_data.info())


##### Visualization 
###### Plot Distribution of Key Columns: Plot histograms or box plots for numerical columns to understand their distributions.
###### Bar Plots for Categorical Data: Visualize the frequency of categories in key categorical columns.
###### Scatter Plots for Relationships: Use scatter plots to identify relationships between key numerical columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to visualize data distributions and relationships
def visualize_data(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 7))
    
    # Plot distribution of full_pallets in order_line
    plt.subplot(2, 2, 1)
    sns.histplot(order_line['full_pallets'], kde=True)
    plt.title('Distribution of Full Pallets')
    
    # Plot distribution of ship_by_date in order_header
    plt.subplot(2, 2, 2)
    sns.histplot(order_header['ship_by_date'], kde=True, bins=30)
    plt.title('Distribution of Ship By Date')
    
    # Plot count of statuses in order_header
    plt.subplot(2, 2, 3)
    sns.countplot(y=order_header['status'])
    plt.title('Count of Statuses in Order Header')
    
    # Plot count of carriers in order_header
    plt.subplot(2, 2, 4)
    sns.countplot(y=order_header['carrier_id'])
    plt.title('Count of Carriers in Order Header')
    
    plt.tight_layout()
    plt.show()

# Visualize the data
visualize_data(order_header, order_line, geo_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to visualize data distributions and relationships
def visualize_data(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 7))
    
    # Plot distribution of full_pallets in order_line (log scale)
    plt.subplot(2, 2, 1)
    sns.histplot(order_line['full_pallets'], kde=True)
    plt.yscale('log')
    plt.title('Distribution of Full Pallets (Log Scale)')
    
    # Plot distribution of ship_by_date in order_header with different line color and horizontal scale in months
    plt.subplot(2, 2, 2)
    sns.histplot(order_header['ship_by_date'], kde=True, bins=30)
    plt.title('Distribution of Ship By Date')
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
    plt.xlabel('Ship By Date 2024')
    
    # Plot count of statuses in order_header (unchanged)
    plt.subplot(2, 2, 3)
    sns.countplot(y=order_header['status'])
    plt.title('Count of Statuses in Order Header')
    
    # Plot count of carriers in order_header with adjusted vertical scale (log scale)
    plt.subplot(2, 2, 4)
    sns.countplot(x=order_header['carrier_id'])
    plt.yscale('log')
    plt.title('Count of Carriers in Order Header')
    plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
    
    plt.tight_layout()
    plt.show()

# Visualize the data
visualize_data(order_header, order_line, geo_data)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Function to create various plots
def create_plots(order_header, order_line, geo_data):
    # Set up the matplotlib figure
    plt.figure(figsize=(14, 10))
    
    # Bar plot: distribution of carriers by full pallets
    plt.subplot(3, 2, 1)
    sns.barplot(x=order_header['carrier_id'], y=order_line['full_pallets'], errorbar=None)
    plt.title('Distribution of Full Pallets by Carrier')
    plt.xlabel('Carrier ID')
    plt.ylabel('Full Pallets')
    plt.xticks(rotation=90)
    
    # Bar plot: count of shipments by status
    plt.subplot(3, 2, 2)
    sns.countplot(x=order_header['status'])
    plt.title('Count of Shipments by Status')
    plt.xlabel('Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    # Heatmap: correlation matrix of numeric features
    plt.subplot(3, 2, 3)
    numeric_features = order_line[['expected_weight', 'full_pallets', 'cpl', 'lpp', 'cpp', 'pallet_type']]
    correlation_matrix = numeric_features.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    plt.title('Correlation Matrix of Numeric Features')
    
    # Box plot: distribution of full pallets by status
    plt.subplot(3, 2, 4)
    sns.boxplot(x=order_header['status'], y=order_line['full_pallets'])
    plt.title('Distribution of Full Pallets by Status')
    plt.xlabel('Status')
    plt.ylabel('Full Pallets')
    plt.xticks(rotation=45)
    
    # Scatter plot: full_pallets vs. ship_by_date with horizontal scale in months
    plt.subplot(3, 2, 5)
    sns.scatterplot(x=order_header['ship_by_date'], y=order_line['full_pallets'], alpha=0.6, edgecolor='w', s=40)
    plt.title('Full Pallets vs. Ship By Date')
    plt.xlabel('2024')
    plt.ylabel('Full Pallets')
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
    
    plt.tight_layout()
    plt.show()

# Generate the plots
create_plots(order_header, order_line, geo_data)


##### Filter Orders Based on Given Criteria:
###### Status: 'Released'
###### Consignment starts with 'Q'
###### Ship by Date: > current time + 4 hours
###### Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']
###### is_vas is 'N'
###### Sum of full pallets per consignment < 45
###### Same customer ID for both load opportunities
###### Sum of full pallets on both load opportunities <= 52

In [ ]:
# Assuming order_header, order_line, geo_data have already been loaded and cleaned

# Step 1: Filter orders based on status 'Released'
def filter_by_status(order_header):
    return order_header[order_header['status'] == 'Released']

# Apply the first filter
filtered_order_header = filter_by_status(order_header)
print("Filtered by Status 'Released':")
filtered_order_header.head()


In [ ]:
# Step 2: Filter orders where consignment starts with 'Q'
def filter_by_consignment(order_header):
    return order_header[order_header['consignment'].str.startswith('Q')]

# Apply the second filter
filtered_order_header = filter_by_consignment(filtered_order_header)
print("Filtered by Consignment starting with 'Q':")
filtered_order_header.head()


In [ ]:
from datetime import datetime, timedelta

# Step 3: Filter orders with ship_by_date > current time + 4 hours
def filter_by_ship_date(order_header):
    current_time_plus_4_hours = datetime.now() + timedelta(hours=4)
    return order_header[order_header['ship_by_date'] > current_time_plus_4_hours]

# Apply the third filter
filtered_order_header = filter_by_ship_date(filtered_order_header)
print("Filtered by Ship by Date > Current Time + 4 Hours:")
print(filtered_order_header.head())


In [ ]:
# Step 4: Filter orders where is_vas is 'N'
def filter_by_is_vas(order_header):
    return order_header[order_header['is_vas'] == 'N']

# Apply the fourth filter
filtered_order_header = filter_by_is_vas(filtered_order_header)
print("Filtered by is_vas 'N':")
print(filtered_order_header.head())


In [ ]:
# Step 5: Filter orders with carrier_id not in the specified list
def filter_by_carrier_id(order_header):
    excluded_carriers = ['SDS', 'DHL', 'CCO', 'MRT']
    return order_header[~order_header['carrier_id'].isin(excluded_carriers)]

# Apply the fifth filter
filtered_order_header = filter_by_carrier_id(filtered_order_header)
print("Filtered by Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']:")
print(filtered_order_header.head())


In [ ]:
# Step 6: Filter consignments with sum of full pallets per consignment < 45
def filter_by_pallet_sums(order_header, order_line):
    merged_data = pd.merge(order_header, order_line, on='order_id')
    consignment_pallet_sums = merged_data.groupby('consignment')['full_pallets'].sum().reset_index()
    return consignment_pallet_sums[consignment_pallet_sums['full_pallets'] < 45]

# Apply the sixth filter
filtered_consignments = filter_by_pallet_sums(filtered_order_header, order_line)
print("Filtered Consignments with Sum of Full Pallets < 45:")
print(filtered_consignments.head())


In [ ]:
# Function to identify consolidation opportunities based on filtered consignments
def identify_consolidation_opportunities(filtered_order_header, filtered_consignments, order_line):
    potential_opportunities = pd.merge(filtered_consignments, filtered_order_header, on='consignment')
    # Ensure only the numeric 'full_pallets' column is summed
    consolidation_opportunities = potential_opportunities.groupby(['customer_id_slim', 'consignment'])[['full_pallets']].sum().reset_index()
    return consolidation_opportunities[consolidation_opportunities['full_pallets'] <= 52]

# Apply the seventh filter and identify consolidation opportunities
filtered_order_header_final = filtered_order_header[filtered_order_header['consignment'].isin(filtered_consignments['consignment'])]
consolidation_opportunities = identify_consolidation_opportunities(filtered_order_header_final, filtered_consignments, order_line)
print("\nConsolidation Opportunities:")
print(consolidation_opportunities)


In [ ]:
# Load geo_data if not already loaded
geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
geo_data = pd.read_excel(geo_data_path)


In [ ]:
import pandas as pd

# Load order data
order_data = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
order_header = order_data['order_header']
order_line = order_data['order_line']

# Load geo data
geo_data = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Standardize customer ID format in geo data
geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.replace('-', '')

# Display the first few rows of each dataframe to understand the structure
print("Order Header:")
print(order_header.head())
print("\nOrder Line:")
print(order_line.head())
print("\nGeo Data:")
print(geo_data.head())


In [ ]:
# import pandas as pd
# from math import radians, cos, sin, sqrt, atan2
# import pyproj

# # Function to convert easting/northing to latitude/longitude
# def easting_northing_to_lat_lon(easting, northing):
#     transformer = pyproj.Transformer.from_crs('epsg:27700', 'epsg:4326')  # British National Grid to WGS84
#     lat, lon = transformer.transform(easting, northing)
#     return lat, lon

# # Load order data
# order_data = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
# order_header = order_data['order_header']
# order_line = order_data['order_line']

# # Load geo data
# geo_data = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# # Standardize customer ID format in geo data
# geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.replace('-', '')

# # Merge order header and order line data on order_id
# order_details = order_header.merge(order_line, on='order_id')

# # Calculate total full pallets per order
# order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')

# # Merge with geo data
# order_details['customer_id_slim_x'] = order_details['customer_id_slim_x'].astype(str)
# merged_orders = order_details.merge(geo_data, left_on='customer_id_slim_x', right_on='CUSTOMER_ID', how='left')

# # Add latitude and longitude columns
# merged_orders[['latitude', 'longitude']] = merged_orders.apply(
#     lambda row: easting_northing_to_lat_lon(row['HOME EASTING'], row['HOME NORTHING']), axis=1, result_type='expand'
# )
# merged_orders = merged_orders.dropna(subset=['latitude', 'longitude'])

# # Function to calculate Haversine distance
# def haversine_distance(lat1, lon1, lat2, lon2):
#     R = 6371.0  # Earth radius in kilometers
#     if -90 <= lat1 <= 90 and -180 <= lon1 <= 180 and -90 <= lat2 <= 90 and -180 <= lon2 <= 180:
#         dlat = radians(lat2 - lat1)
#         dlon = radians(lon2 - lon1)
#         a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
#         c = 2 * atan2(sqrt(a), sqrt(1 - a))
#         distance = R * c
#         return distance
#     else:
#         return None

# opportunities = []
# for i, row1 in merged_orders.iterrows():
#     for j, row2 in merged_orders.iterrows():
#         if i >= j:
#             continue
#         # Ensure valid coordinates
#         if pd.notna(row1['latitude']) and pd.notna(row1['longitude']) and pd.notna(row2['latitude']) and pd.notna(row2['longitude']):
#             distance = haversine_distance(row1['latitude'], row1['longitude'], row2['latitude'], row2['longitude'])
#             if distance is not None and distance <= 48: # 30 miles is approximately 48 kilometers
#                 combined_pallets = row1['total_full_pallets'] + row2['total_full_pallets']
#                 opportunities.append((row1['order_id'], row2['order_id'], combined_pallets, distance))

# # Convert opportunities to DataFrame for better visualization
# opportunities_df = pd.DataFrame(opportunities, columns=['Order 1', 'Order 2', 'Combined Pallets', 'Distance (km)'])

# # Group by 'Order 1' and 'Order 2' to remove duplicate pairs
# opportunities_df = opportunities_df.groupby(['Order 1', 'Order 2']).agg({'Combined Pallets': 'first', 'Distance (km)': 'first'}).reset_index()

# # Save the opportunities DataFrame to an Excel file for easy review
# opportunities_df.to_excel(os.path.join(DATA_DIR, 'consolidation_opportunities_coordinates_postcodes.xlsx'), index=False)

# print("Consolidation opportunities have been saved to os.path.join(DATA_DIR, 'consolidation_opportunities_coordinates_postcodes.xlsx').")

##### Approach 2: Using Google Maps API
###### Geocoding:
###### Postcodes are geocoded using the Google Maps API to obtain latitude/longitude coordinates.
###### This approach provides accurate geolocation data, leveraging Google's robust mapping service.
###### Distance Calculation and Opportunity Identification:
###### Similar to the first approach, the haversine formula is used for distance calculation.
###### Opportunities are identified by iterating through pairs of geocoded orders.

In [ ]:
import pandas as pd
import requests
from math import radians, cos, sin, sqrt, atan2

# Load order data
order_data = pd.read_excel(os.path.join(DATA_DIR, SYS_ORDER_FILE), sheet_name=None)
order_header = order_data['order_header']
order_line = order_data['order_line']

# Load geo data
geo_data = pd.read_excel(os.path.join(DATA_DIR, GEO_FILE))

# Standardize customer ID format in geo data
geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.replace('-', '')

# Merge order header and order line data on order_id
order_details = order_header.merge(order_line, on='order_id')

# Calculate total full pallets per order
order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')

# Function to geocode address using Google Maps API
def geocode_postcode(postcode, maps_key):
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": postcode, "key": maps_key}
    response = requests.get(base_url, params=params)
    geo_data = response.json()
    if geo_data['status'] == 'OK':
        location = geo_data['results'][0]['geometry']['location']
        return (location['lat'], location['lng'])
    else:
        return (None, None)

# Your Google Maps API key
maps_key = "<REDACTED_KEY>"

# Geocode each postcode in the order details
order_details['lat_long'] = order_details['postcode'].apply(lambda x: geocode_postcode(x, maps_key))

# Separate latitude and longitude into different columns
order_details[['latitude', 'longitude']] = pd.DataFrame(order_details['lat_long'].tolist(), index=order_details.index, columns=['latitude', 'longitude'])

# Drop rows with missing latitude or longitude
order_details = order_details.dropna(subset=['latitude', 'longitude'])

# Function to calculate haversine distance
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Identify nearby consolidation opportunities
opportunities = []
for i, row1 in order_details.iterrows():
    for j, row2 in order_details.iterrows():
        if i >= j:
            continue
        distance = haversine_distance(row1['latitude'], row1['longitude'], row2['latitude'], row2['longitude'])
        if distance <= 48:  # 30 miles is approximately 48 kilometers
            combined_pallets = row1['total_full_pallets'] + row2['total_full_pallets']
            opportunities.append((row1['order_id'], row2['order_id'], combined_pallets, distance))

# Convert opportunities to DataFrame for better visualization
opportunities_df = pd.DataFrame(opportunities, columns=['Order 1', 'Order 2', 'Combined Pallets', 'Distance (km)'])

# Group by 'Order 1' and 'Order 2' to remove duplicate pairs
opportunities_df = opportunities_df.groupby(['Order 1', 'Order 2']).agg({'Combined Pallets': 'first', 'Distance (km)': 'first'}).reset_index()

# Display the DataFrame
print("Consolidation Opportunities using Google Maps API:")
print(opportunities_df)

# Save the opportunities DataFrame to an Excel file for easy review
opportunities_df.to_excel(os.path.join(DATA_DIR, 'consolidation_opportunities_google_maps.xlsx'), index=False)

print("Consolidation opportunities have been saved to os.path.join(DATA_DIR, 'consolidation_opportunities_google_maps.xlsx').")

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot of orders with consolidation opportunities
plt.figure(figsize=(10, 6))
plt.scatter(order_details['longitude'], order_details['latitude'], c='blue', label='Orders', alpha=0.5)
for index, row in opportunities_df.iterrows():
    order1 = order_details[order_details['order_id'] == row['Order 1']].iloc[0]
    order2 = order_details[order_details['order_id'] == row['Order 2']].iloc[0]
    plt.plot([order1['longitude'], order2['longitude']], [order1['latitude'], order2['latitude']], 'ro-', label='Consolidation Opportunity' if index == 0 else "", alpha=0.7)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Scatter Plot of Orders with Consolidation Opportunities')
plt.legend()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Aggregate data by ranges or simplify x-axis labels
opportunities_df['Order Pair'] = opportunities_df['Order 1'].astype(str) + " & " + opportunities_df['Order 2'].astype(str)

# Sort data by combined pallets to show the top opportunities
top_opportunities_df = opportunities_df.sort_values(by='Combined Pallets', ascending=False).head(20)

# Bar plot of combined pallets for top consolidation opportunities
plt.figure(figsize=(14, 8))
sns.barplot(data=top_opportunities_df, x='Order Pair', y='Combined Pallets', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 Combined Pallets for Consolidation Opportunities')
plt.xlabel('Order Pairs')
plt.ylabel('Combined Pallets')
plt.tight_layout()
plt.show()


In [ ]:
# Histogram of distances between consolidation opportunities
plt.figure(figsize=(10, 6))
sns.histplot(opportunities_df['Distance (km)'], bins=30, kde=True)
plt.title('Histogram of Distances Between Consolidation Opportunities')
plt.xlabel('Distance (km)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# import folium
# from folium.plugins import MarkerCluster

# # Create a map centered around the average location of the orders
# m = folium.Map(location=[order_details['latitude'].mean(), order_details['longitude'].mean()], zoom_start=10)

# # Add orders to the map
# marker_cluster = MarkerCluster().add_to(m)
# for idx, row in order_details.iterrows():
#     folium.Marker([row['latitude'], row['longitude']], popup=f"Order ID: {row['order_id']}").add_to(marker_cluster)

# # Add consolidation opportunities to the map
# for idx, row in opportunities_df.iterrows():
#     order1 = order_details[order_details['order_id'] == row['Order 1']].iloc[0]
#     order2 = order_details[order_details['order_id'] == row['Order 2']].iloc[0]
#     folium.PolyLine([(order1['latitude'], order1['longitude']), (order2['latitude'], order2['longitude'])], color='red', weight=2.5, opacity=0.8).add_to(m)

# # Save and display the map
# m.save(os.path.join(DATA_DIR, 'consolidation_opportunities_map.html'))
# m


In [ ]:
import folium
from folium.plugins import MarkerCluster
import requests

def geocode_postcode(postcode, maps_key):
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": postcode, "key": maps_key}
    response = requests.get(base_url, params=params)
    geo_data = response.json()
    if geo_data['status'] == 'OK':
        location = geo_data['results'][0]['geometry']['location']
        return (location['lat'], location['lng'])
    else:
        return (None, None)

# Your Google Maps API key
maps_key = "<REDACTED_KEY>"

# Geocode postcodes and update latitude and longitude values
order_details['lat_long'] = order_details['postcode'].apply(lambda x: geocode_postcode(x, maps_key))
order_details[['latitude', 'longitude']] = pd.DataFrame(order_details['lat_long'].tolist(), index=order_details.index, columns=['latitude', 'longitude'])
order_details = order_details.dropna(subset=['latitude', 'longitude'])

# Create a map centered around the average location of the orders
m = folium.Map(location=[order_details['latitude'].mean(), order_details['longitude'].mean()], zoom_start=10)

# Add orders to the map
marker_cluster = MarkerCluster().add_to(m)
for idx, row in order_details.iterrows():
    folium.Marker([row['latitude'], row['longitude']], popup=f"Order ID: {row['order_id']}, Postcode: {row['postcode']}").add_to(marker_cluster)

# Add consolidation opportunities to the map
for idx, row in opportunities_df.iterrows():
    order1 = order_details[order_details['order_id'] == row['Order 1']].iloc[0]
    order2 = order_details[order_details['order_id'] == row['Order 2']].iloc[0]
    folium.PolyLine([(order1['latitude'], order1['longitude']), (order2['latitude'], order2['longitude'])], color='red', weight=2.5, opacity=0.8).add_to(m)

# Save and display the map
m.save(os.path.join(DATA_DIR, 'consolidation_opportunities_map2.html'))
m

In [ ]:
## Adjust or remove some of the filters to see if more consolidation opportunities emerge
## The only filters applied are: he identify_consolidation_opportunities_no_filters() 
##  function merges the order_header and order_line DataFrames on order_id.
## It calculates the total number of full pallets per order.
## It groups the merged DataFrame by customer_id_slim_x (customer ID) and consignment 
##  to identify potential consolidation opportunities based on the sum of full pallets.

import pandas as pd

# Load the original order data again
def load_data():
    order_data_path = os.path.join(DATA_DIR, SYS_ORDER_FILE)
    geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
    
    order_data = pd.read_excel(order_data_path, sheet_name=None)
    geo_data = pd.read_excel(geo_data_path)
    
    order_header = order_data['order_header']
    order_line = order_data['order_line']
    
    return order_header, order_line, geo_data

# Clean the data
def clean_data(order_header, order_line, geo_data):
    order_header = order_header.dropna(subset=['status', 'consignment', 'ship_by_date', 'is_vas', 'carrier_id', 'customer_id_slim'])
    order_line = order_line.dropna(subset=['order_id', 'full_pallets'])
    geo_data = geo_data.dropna()
    order_header.loc[:, 'ship_by_date'] = pd.to_datetime(order_header['ship_by_date'])
    
    return order_header, order_line, geo_data

# Standardize customer ID format in geo data
def standardize_formats(order_header, order_line, geo_data):
    order_header['postcode'] = order_header['postcode'].str.upper().str.replace(' ', '')
    order_line['customer_id'] = order_line['customer_id'].str.upper().str.strip()
    geo_data['POSTCODE'] = geo_data['POSTCODE'].str.upper().str.replace(' ', '')
    geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.upper().str.strip()
    return order_header, order_line, geo_data

# Merge order header and order line data on order_id
def merge_order_data(order_header, order_line, geo_data):
    order_details = order_header.merge(order_line, on='order_id')
    order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')
    order_details['customer_id_slim_x'] = order_details['customer_id_slim_x'].astype(str)
    merged_orders = order_details.merge(geo_data, left_on='customer_id_slim_x', right_on='CUSTOMER_ID', how='left')
    return merged_orders

# Function to identify consolidation opportunities based on total full pallets
def identify_consolidation_opportunities_no_filters(order_header, order_line):
    # Merge order header and order line data on order_id
    order_details = order_header.merge(order_line, on='order_id')
    
    # Calculate total full pallets per order
    order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')
    
    # Identify opportunities by grouping on customer_id and consignment
    potential_opportunities = order_details.groupby(['customer_id_slim_x', 'consignment'])[['full_pallets']].sum().reset_index()
    
    # Return the consolidation opportunities
    return potential_opportunities

# Load the data
order_header, order_line, geo_data = load_data()

# Clean the data
order_header, order_line, geo_data = clean_data(order_header, order_line, geo_data)

# Standardize formats
order_header, order_line, geo_data = standardize_formats(order_header, order_line, geo_data)

# Identify consolidation opportunities without any filters
consolidation_opportunities_no_filters = identify_consolidation_opportunities_no_filters(order_header, order_line)

# Display the consolidation opportunities
print("Consolidation Opportunities without Filters:")
print(consolidation_opportunities_no_filters)


In [ ]:
# def apply_filters_and_identify_opportunities(order_header, order_line):
#     filtered_order_header = filter_by_status(order_header)
#     print("Consolidation Opportunities with Status 'Released':")
#     print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     filtered_order_header = filter_by_consignment(filtered_order_header)
#     print("Consolidation Opportunities with Consignment starting with 'Q':")
#     print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     # filtered_order_header = filter_by_ship_date(filtered_order_header)
#     # print("Consolidation Opportunities with Ship by Date > Current Time + 4 Hours:")
#     # print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     filtered_order_header = filter_by_ship_date(filtered_order_header)
#     print("Consolidation Opportunities with Ship by Date > Current Time + 4 Hours:")
#     print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     filtered_order_header = filter_by_is_vas(filtered_order_header)
#     print("Consolidation Opportunities with is_vas 'N':")
#     print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     filtered_order_header = filter_by_carrier_id(filtered_order_header)
#     print("Consolidation Opportunities with Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']:")
#     print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
#     valid_consignments = filter_by_pallet_sums(filtered_order_header, order_line)
#     filtered_order_header = filtered_order_header[filtered_order_header['consignment'].isin(valid_consignments['consignment'])]
#     consolidation_opportunities_pallet_sums = identify_consolidation_opportunities_no_filters(filtered_order_header, order_line)
#     print("Consolidation Opportunities with Sum of Full Pallets per Consignment < 45:")
#     print(consolidation_opportunities_pallet_sums)
    
#     consolidation_opportunities_combined_pallet_sums = filter_by_combined_pallet_sums(consolidation_opportunities_pallet_sums)
#     print("Consolidation Opportunities with Sum of Full Pallets on Both Load Opportunities <= 52:")
#     print(consolidation_opportunities_combined_pallet_sums)

# # Apply all filters and identify opportunities step by step
# apply_filters_and_identify_opportunities(order_header, order_line)


In [ ]:
## Adjust or remove some of the filters to see if more consolidation opportunities emerge
## The only filters applied are: he identify_consolidation_opportunities_no_filters() 
##  function merges the order_header and order_line DataFrames on order_id.
## It calculates the total number of full pallets per order.
## It groups the merged DataFrame by customer_id_slim_x (customer ID) and consignment 
##  to identify potential consolidation opportunities based on the sum of full pallets.

import pandas as pd

# Load the original order data again
def load_data():
    order_data_path = os.path.join(DATA_DIR, SYS_ORDER_FILE)
    geo_data_path = os.path.join(DATA_DIR, GEO_FILE)
    
    order_data = pd.read_excel(order_data_path, sheet_name=None)
    geo_data = pd.read_excel(geo_data_path)
    
    order_header = order_data['order_header']
    order_line = order_data['order_line']
    
    return order_header, order_line, geo_data

# Clean the data
def clean_data(order_header, order_line, geo_data):
    order_header = order_header.dropna(subset=['status', 'consignment', 'ship_by_date', 'is_vas', 'carrier_id', 'customer_id_slim'])
    order_line = order_line.dropna(subset=['order_id', 'full_pallets'])
    geo_data = geo_data.dropna()
    order_header.loc[:, 'ship_by_date'] = pd.to_datetime(order_header['ship_by_date'])
    
    return order_header, order_line, geo_data

# Standardize customer ID format in geo data
def standardize_formats(order_header, order_line, geo_data):
    order_header['postcode'] = order_header['postcode'].str.upper().str.replace(' ', '')
    order_line['customer_id'] = order_line['customer_id'].str.upper().str.strip()
    geo_data['POSTCODE'] = geo_data['POSTCODE'].str.upper().str.replace(' ', '')
    geo_data['CUSTOMER_ID'] = geo_data['CUSTOMER_ID'].str.upper().str.strip()
    return order_header, order_line, geo_data

# Merge order header and order line data on order_id
def merge_order_data(order_header, order_line, geo_data):
    order_details = order_header.merge(order_line, on='order_id')
    order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')
    order_details['customer_id_slim_x'] = order_details['customer_id_slim_x'].astype(str)
    merged_orders = order_details.merge(geo_data, left_on='customer_id_slim_x', right_on='CUSTOMER_ID', how='left')
    return merged_orders

# Function to identify consolidation opportunities based on total full pallets
def identify_consolidation_opportunities_no_filters(order_header, order_line):
    # Merge order header and order line data on order_id
    order_details = order_header.merge(order_line, on='order_id')
    
    # Calculate total full pallets per order
    order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')
    
    # Identify opportunities by grouping on customer_id and consignment
    potential_opportunities = order_details.groupby(['customer_id_slim_x', 'consignment'])[['full_pallets']].sum().reset_index()
    
    # Return the consolidation opportunities
    return potential_opportunities

# Load the data
order_header, order_line, geo_data = load_data()

# Clean the data
order_header, order_line, geo_data = clean_data(order_header, order_line, geo_data)

# Standardize formats
order_header, order_line, geo_data = standardize_formats(order_header, order_line, geo_data)

# Identify consolidation opportunities without any filters
consolidation_opportunities_no_filters = identify_consolidation_opportunities_no_filters(order_header, order_line)

# Display the consolidation opportunities
print("Consolidation Opportunities without Filters:")
print(consolidation_opportunities_no_filters)


##### A different approach to confirm if the ship_date filter ends up cancelling out all other opportunties.
###### This was confirmed as you can not once the filter is applied all subsequent dataframes are blanks

In [ ]:
import pandas as pd
from datetime import timedelta

# Function to filter by status
def filter_by_status(order_header):
    return order_header[order_header['status'] == 'Released']

# Function to filter by consignment
def filter_by_consignment(order_header):
    return order_header[order_header['consignment'].str.startswith('Q')]

# Function to filter by ship date
def filter_by_ship_date(order_header, reference_date):
    return order_header[order_header['ship_by_date'] > reference_date + timedelta(hours=4)]

# Function to filter by is_vas
def filter_by_is_vas(order_header):
    return order_header[order_header['is_vas'] == 'N']

# Function to filter by carrier ID
def filter_by_carrier_id(order_header):
    excluded_carriers = ['SDS', 'DHL', 'CCO', 'MRT']
    return order_header[~order_header['carrier_id'].isin(excluded_carriers)]

# Function to filter by pallet sums
def filter_by_pallet_sums(order_header, order_line):
    merged_data = pd.merge(order_header, order_line, on='order_id')
    consignment_pallet_sums = merged_data.groupby('consignment')['full_pallets'].sum().reset_index()
    return consignment_pallet_sums[consignment_pallet_sums['full_pallets'] < 45]

# Function to identify consolidation opportunities without any filters
def identify_consolidation_opportunities_no_filters(order_header, order_line):
    order_details = order_header.merge(order_line, on='order_id')
    order_details['total_full_pallets'] = order_details.groupby('order_id')['full_pallets'].transform('sum')
    potential_opportunities = order_details.groupby(['customer_id_slim_x', 'consignment'])[['full_pallets']].sum().reset_index()
    return potential_opportunities

# Apply filters step-by-step and identify opportunities
def apply_filters_and_identify_opportunities(order_header, order_line):
    # Identify the latest date in the dataset
    latest_date = order_header['ship_by_date'].max()
    print("Latest date in the dataset:", latest_date)

    filtered_order_header = filter_by_status(order_header)
    print("Consolidation Opportunities with Status 'Released':")
    print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
    filtered_order_header = filter_by_consignment(filtered_order_header)
    print("Consolidation Opportunities with Consignment starting with 'Q':")
    print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
    filtered_order_header = filter_by_ship_date(filtered_order_header, latest_date)
    print("Consolidation Opportunities with Ship by Date > Latest Date + 4 Hours:")
    print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
    filtered_order_header = filter_by_is_vas(filtered_order_header)
    print("Consolidation Opportunities with is_vas 'N':")
    print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
    filtered_order_header = filter_by_carrier_id(filtered_order_header)
    print("Consolidation Opportunities with Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']:")
    print(identify_consolidation_opportunities_no_filters(filtered_order_header, order_line))
    
    valid_consignments = filter_by_pallet_sums(filtered_order_header, order_line)
    filtered_order_header = filtered_order_header[filtered_order_header['consignment'].isin(valid_consignments['consignment'])]
    consolidation_opportunities_pallet_sums = identify_consolidation_opportunities_no_filters(filtered_order_header, order_line)
    print("Consolidation Opportunities with Sum of Full Pallets per Consignment < 45:")
    print(consolidation_opportunities_pallet_sums)
    
    consolidation_opportunities_combined_pallet_sums = filter_by_combined_pallet_sums(consolidation_opportunities_pallet_sums)
    print("Consolidation Opportunities with Sum of Full Pallets on Both Load Opportunities <= 52:")
    print(consolidation_opportunities_combined_pallet_sums)

# Apply all filters and identify opportunities step by step
apply_filters_and_identify_opportunities(order_header, order_line)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Function to create a heatmap of order densities
def create_heatmap(order_header):
    # Extract relevant columns for the heatmap
    heatmap_data = order_header[['ship_by_date', 'customer_id_slim']]
    
    # Create a pivot table to count the number of orders per customer per ship_by_date
    pivot_table = heatmap_data.pivot_table(index='ship_by_date', columns='customer_id_slim', aggfunc='size', fill_value=0)
    
    # Create a heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_table, cmap='YlGnBu', linewidths=0.5)
    plt.title('Heatmap of Orders per Customer by Ship By Date')
    plt.xlabel('Customer ID')
    plt.ylabel('Ship By Date')
    plt.show()

# Load the data
order_header, order_line, geo_data = load_data()

# Clean the data
order_header, order_line, geo_data = clean_data(order_header, order_line, geo_data)

# Standardize formats
order_header, order_line, geo_data = standardize_formats(order_header, order_line, geo_data)

# Create the heatmap
create_heatmap(order_header)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Function to create a daily aggregated heatmap of order densities
def create_daily_heatmap(order_header):
    # Extract relevant columns for the heatmap
    order_header['ship_by_date'] = order_header['ship_by_date'].dt.date
    heatmap_data = order_header[['ship_by_date', 'customer_id_slim']]
    
    # Create a pivot table to count the number of orders per customer per ship_by_date
    pivot_table = heatmap_data.pivot_table(index='ship_by_date', columns='customer_id_slim', aggfunc='size', fill_value=0)
    
    # Create a heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot_table, cmap='YlGnBu', linewidths=0.5)
    plt.title('Daily Heatmap of Orders per Customer by Ship By Date')
    plt.xlabel('Customer ID')
    plt.ylabel('Ship By Date')
    plt.show()

# Create the daily heatmap
create_daily_heatmap(order_header)


##### Priority: Medium
###### 3. Establish a way for the colleague to extract and regularly review all the current rules applied to ensure 
###### they remain valid over time.

In [ ]:
# Mock interface function to send opportunities to a colleague and receive feedback
def send_opportunities_to_colleague(opportunities_df):
    # For now, we'll simulate the feedback process with a simple print statement and input
    for index, row in opportunities_df.iterrows():
        print(f"Opportunity: Combine Order {row['Order 1']} and Order {row['Order 2']}")
        print(f"Combined Pallets: {row['Combined Pallets']}, Distance: {row['Distance (km)']:.2f} km")
        feedback = input("Is this opportunity feasible? (yes/no): ")
        if feedback.lower() == 'no':
            reason = input("Why is this opportunity not feasible? ")
            # Log the feedback for further analysis
            print(f"Feedback: {reason}")
        print("\n")
        
# Sample opportunities DataFrame for demonstration
sample_opportunities_df = pd.DataFrame({
    'Order 1': [101, 102],
    'Order 2': [201, 202],
    'Combined Pallets': [30, 35],
    'Distance (km)': [40, 45]
})

# Send the sample opportunities to a colleague for feedback
send_opportunities_to_colleague(sample_opportunities_df)

# Mock function to extract and review all current rules applied
def extract_and_review_rules():
    # List of current rules applied
    rules = [
        "Order status must be 'Released'",
        "Consignment must start with 'Q'",
        "Ship by Date must be > current time + 4 hours",
        "is_vas must be 'N'",
        "Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']",
        "Sum of full pallets per consignment < 45",
        "Same customer ID for both load opportunities",
        "Sum of full pallets on both load opportunities <= 52"
    ]
    
    # Display the current rules for review
    print("Current Rules Applied:")
    for rule in rules:
        print(f"- {rule}")
        
    # Simulate the review process
    print("\nReviewing Rules:")
    for rule in rules:
        print(f"Is this rule still valid? (yes/no): {rule}")
        feedback = input()
        if feedback.lower() == 'no':
            reason = input("Why is this rule no longer valid? ")
            # Log the feedback for further analysis
            print(f"Feedback: {reason}")
        print("\n")

# Extract and review the current rules
extract_and_review_rules()

In [ ]:
# Mock function to extract and review all current rules applied
def extract_and_review_rules():
    # List of current rules applied
    rules = [
        "Order status must be 'Released'",
        "Consignment must start with 'Q'",
        "Ship by Date must be > current time + 4 hours",
        "is_vas must be 'N'",
        "Carrier ID not in ['SDS', 'DHL', 'CCO', 'MRT']",
        "Sum of full pallets per consignment < 45",
        "Same customer ID for both load opportunities",
        "Sum of full pallets on both load opportunities <= 52"
    ]
    
    # Display the current rules for review
    print("Current Rules Applied:")
    for rule in rules:
        print(f"- {rule}")
        
    # Simulate the review process
    print("\nReviewing Rules:")
    for rule in rules:
        print(f"Is this rule still valid? (yes/no): {rule}")
        feedback = input()
        if feedback.lower() == 'no':
            reason = input("Why is this rule no longer valid? ")
            # Log the feedback for further analysis
            print(f"Feedback: {reason}")
        print("\n")

# Extract and review the current rules
extract_and_review_rules()
